# Explore Luxe Lab Parquet Results

This notebook loads every `.parquet` file from a Luxe Lab build output and shows row counts, column types, and sample rows.

**Default run folder:** `data/my-first-run` (change `RUN_DIR` below if you built to a different path).

In [1]:
from pathlib import Path
import json
import html

import duckdb
from IPython.display import HTML, Markdown, display

try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False

# Point this at any luxe_lab build output folder
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "luxe_lab").exists() and (PROJECT_ROOT.parent / "luxe_lab").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RUN_DIR = PROJECT_ROOT / "data" / "my-first-run"
MANIFEST_PATH = RUN_DIR / "manifest.json"
con = duckdb.connect()

print(f"Project root: {PROJECT_ROOT}")
print(f"Run directory: {RUN_DIR}")
print(f"Exists: {RUN_DIR.exists()}")
print(f"Pandas available: {HAS_PANDAS}")


def preview_table(path: Path, limit: int = 10):
    result = con.execute(
        "SELECT * FROM read_parquet(?) LIMIT ?",
        [str(path), limit],
    )
    columns = [col[0] for col in result.description]
    rows = result.fetchall()

    if HAS_PANDAS:
        display(pd.DataFrame(rows, columns=columns))
        return

    table_html = "<table><thead><tr>" + "".join(
        f"<th>{html.escape(str(col))}</th>" for col in columns
    ) + "</tr></thead><tbody>"
    for row in rows:
        table_html += "<tr>" + "".join(
            f"<td>{html.escape('' if value is None else str(value))}</td>" for value in row
        ) + "</tr>"
    table_html += "</tbody></table>"
    display(HTML(table_html))

Project root: C:\Users\visha\Documents\Codex\Projects\luxe-cafe-paid-media-decision-lab
Run directory: C:\Users\visha\Documents\Codex\Projects\luxe-cafe-paid-media-decision-lab\data\my-first-run
Exists: True
Pandas available: True


In [2]:
if MANIFEST_PATH.exists():
    with MANIFEST_PATH.open() as f:
        manifest = json.load(f)

    display(Markdown(
        f"**Status:** `{manifest['status']}`  \n"
        f"**Dataset SHA256:** `{manifest['dataset_sha256'][:16]}…`  \n"
        f"**Report window:** {manifest['config']['report_start']} → {manifest['config']['report_end']}"
    ))
else:
    manifest = None
    print("No manifest.json found — will still list parquet files on disk.")

**Status:** `validated`  
**Dataset SHA256:** `a58df509f2034733…`  
**Report window:** 2026-05-04 → 2026-08-23

In [3]:
parquet_files = sorted(RUN_DIR.rglob("*.parquet"))
sections = {}

for path in parquet_files:
    section = path.parent.name  # e.g. raw, processed
    sections.setdefault(section, []).append(path)

summary_rows = []
for section, paths in sorted(sections.items()):
    for path in sorted(paths):
        row_count = con.execute(
            "SELECT COUNT(*) FROM read_parquet(?)", [str(path)]
        ).fetchone()[0]
        summary_rows.append({
            "section": section,
            "table": path.stem,
            "rows": row_count,
            "path": str(path.relative_to(PROJECT_ROOT)),
        })

if HAS_PANDAS:
    display(pd.DataFrame(summary_rows))
else:
    display(HTML(
        "<table><thead><tr><th>section</th><th>table</th><th>rows</th><th>path</th></tr></thead><tbody>"
        + "".join(
            f"<tr><td>{r['section']}</td><td>{r['table']}</td><td>{r['rows']}</td><td>{r['path']}</td></tr>"
            for r in summary_rows
        )
        + "</tbody></table>"
    ))

,section,table,rows,path
0,processed,campaign_daily,749,data\my-first-run\processed\campaign_daily.par...
1,processed,order_attribution,1846,data\my-first-run\processed\order_attribution....
2,processed,order_economics,1846,data\my-first-run\processed\order_economics.pa...
3,processed,platform_latest,672,data\my-first-run\processed\platform_latest.pa...
4,processed,reconciliation_daily,112,data\my-first-run\processed\reconciliation_dai...
5,raw,budgets,852,data\my-first-run\raw\budgets.parquet
6,raw,campaigns,6,data\my-first-run\raw\campaigns.parquet
7,raw,costs,152,data\my-first-run\raw\costs.parquet
8,raw,creatives,12,data\my-first-run\raw\creatives.parquet
9,raw,inventory,142,data\my-first-run\raw\inventory.parquet


## Table previews

Each section below shows the first 10 rows and column dtypes for every parquet file.

In [4]:
PREVIEW_ROWS = 10

for section, paths in sorted(sections.items()):
    display(Markdown(f"### {section}"))

    for path in sorted(paths):
        row_count = con.execute(
            "SELECT COUNT(*) FROM read_parquet(?)", [str(path)]
        ).fetchone()[0]

        display(Markdown(f"#### `{path.stem}` — {row_count:,} rows"))
        preview_table(path, limit=PREVIEW_ROWS)

        if HAS_PANDAS:
            df = con.execute("SELECT * FROM read_parquet(?)", [str(path)]).df()
            display(df.dtypes.to_frame("dtype"))

### processed

#### `campaign_daily` — 749 rows

,report_date,campaign_id,impressions,clicks,spend_cents,media_present,orders,net_revenue_cents,booked_revenue_cents,missing_cost_lines,known_contribution_cents,contribution_before_media_cents,contribution_after_media_cents,business_net_roas,ctr,cpc_usd,planned_spend_cents,pacing_ratio,source_class
0,2026-05-04,C01,844.0,26.0,4940,True,2,119998,119998,0,59058,59058,54118,24.291093,0.030806,1.900000,4180.0,1.181818,derived_result
1,2026-05-04,C02,1206.0,30.0,8100,True,3,119998,179997,0,48988,48988,40888,14.814568,0.024876,2.700000,5940.0,1.363636,derived_result
2,2026-05-04,C03,1332.0,28.0,6440,True,2,59999,119998,0,19459,19459,13019,9.316615,0.021021,2.300000,5060.0,1.272727,derived_result
3,2026-05-04,C04,1204.0,28.0,4760,True,1,53999,53999,0,23703,23703,18943,11.344328,0.023256,1.700000,3740.0,1.272727,derived_result
4,2026-05-04,C05,1387.0,29.0,4640,True,3,179997,179997,0,88587,88587,83947,38.792457,0.020908,1.600000,3520.0,1.318182,derived_result
5,2026-05-04,C06,NaN,27.0,3599,True,1,59999,59999,0,29529,29529,25930,16.671020,NaN,1.332963,5000.0,0.719800,derived_result
6,2026-05-04,UNATTRIBUTED,NaN,NaN,0,False,2,119998,119998,0,59058,59058,59058,NaN,NaN,NaN,NaN,NaN,derived_result
7,2026-05-05,C01,1103.0,24.0,4560,True,3,239996,239996,0,118146,118146,113586,52.630702,0.021759,1.900000,4180.0,1.090909,derived_result
8,2026-05-05,C02,757.0,25.0,6750,True,1,59999,59999,0,29529,29529,22779,8.888741,0.033025,2.700000,5940.0,1.136364,derived_result
9,2026-05-05,C03,1177.0,32.0,7360,True,4,239996,239996,0,118116,118116,110756,32.608152,0.027188,2.300000,5060.0,1.454545,derived_result


,dtype
report_date,datetime64[us]
campaign_id,str
impressions,Int64
clicks,Int64
spend_cents,int64
media_present,bool
orders,int64
net_revenue_cents,int64
booked_revenue_cents,int64
missing_cost_lines,int64


#### `order_attribution` — 1,846 rows

...

,refund_id,order_id,line_id,event_at,available_at,refund_cents,returned_quantity,recovered_inventory_cents,handling_cents,fee_refund_cents,source_class
0,R00001,O000003,1,2026-05-04 09:25:00-04:00,2026-05-04 14:25:00-04:00,59999,1,21600,1200,0,synthetic_observation
1,R00002,O000005,1,2026-05-10 15:42:00-04:00,2026-05-10 20:42:00-04:00,59999,1,21600,1200,0,synthetic_observation
2,R00003,O000018,1,2026-04-15 11:47:00-04:00,2026-04-15 16:47:00-04:00,53999,1,21600,1200,0,synthetic_observation
3,R00004,O000028,1,2026-05-05 16:40:00-04:00,2026-05-05 21:40:00-04:00,59999,1,21600,1200,0,synthetic_observation
4,R00005,O000044,1,2026-05-10 15:59:00-04:00,2026-05-10 20:59:00-04:00,59999,1,21600,1200,0,synthetic_observation
5,R00006,O000058,1,2026-04-23 16:05:00-04:00,2026-04-23 21:05:00-04:00,59999,1,21600,1200,0,synthetic_observation
6,R00007,O000072,1,2026-05-10 14:01:00-04:00,2026-05-10 19:01:00-04:00,59999,1,21600,1200,0,synthetic_observation
7,R00008,O000087,1,2026-05-01 09:46:00-04:00,2026-05-01 14:46:00-04:00,59999,1,21600,1200,0,synthetic_observation
8,R00009,O000095,1,2026-04-28 16:06:00-04:00,2026-04-28 21:06:00-04:00,59999,1,21600,1200,0,synthetic_observation
9,R00010,O000097,1,2026-05-10 09:14:00-04:00,2026-05-10 14:14:00-04:00,59999,1,21600,1200,0,synthetic_observation


,dtype
refund_id,str
order_id,str
line_id,str
event_at,"datetime64[us, America/New_York]"
available_at,"datetime64[us, America/New_York]"
refund_cents,int64
returned_quantity,int32
recovered_inventory_cents,int64
handling_cents,int64
fee_refund_cents,int64


#### `sessions` — 25,034 rows

,session_id,journey_id,customer_id,campaign_id,ad_id,event_at,available_at,source_class
0,S0000001,J0000001,U0000001,C01,C01-A0,2026-04-04 19:17:00-04:00,2026-04-04 20:17:00-04:00,synthetic_observation
1,S0000002,J0000002,U0000002,C01,C01-A0,2026-04-04 18:47:00-04:00,2026-04-04 19:47:00-04:00,synthetic_observation
2,S0000003,J0000003,U0000001,C01,C01-A0,2026-04-04 11:14:00-04:00,2026-04-04 12:14:00-04:00,synthetic_observation
3,S0000004,J0000004,U0000004,C01,C01-A1,2026-04-04 19:34:00-04:00,2026-04-04 20:34:00-04:00,synthetic_observation
4,S0000005,J0000005,U0000005,C01,C01-A0,2026-04-04 08:48:00-04:00,2026-04-04 09:48:00-04:00,synthetic_observation
5,S0000006,J0000006,U0000006,C01,C01-A1,2026-04-04 10:13:00-04:00,2026-04-04 11:13:00-04:00,synthetic_observation
6,S0000007,J0000007,U0000005,C01,C01-A1,2026-04-04 16:07:00-04:00,2026-04-04 17:07:00-04:00,synthetic_observation
7,S0000008,J0000008,U0000001,C01,C01-A0,2026-04-04 08:42:00-04:00,2026-04-04 09:42:00-04:00,synthetic_observation
8,S0000009,J0000009,U0000009,C01,C01-A0,2026-04-04 11:55:00-04:00,2026-04-04 12:55:00-04:00,synthetic_observation
9,S0000010,J0000010,U0000010,C01,C01-A1,2026-04-04 13:10:00-04:00,2026-04-04 14:10:00-04:00,synthetic_observation


,dtype
session_id,str
journey_id,str
customer_id,str
campaign_id,str
ad_id,str
event_at,"datetime64[us, America/New_York]"
available_at,"datetime64[us, America/New_York]"
source_class,str
